# Cheminformatics: Molecular Fingerprints, Similarity & QSAR with RDKit

**Author:** Nandini  
**Stack:** RDKit, scikit-learn, pandas, matplotlib  
**Topics:** Morgan/ECFP fingerprints, Tanimoto similarity, Murcko scaffolds, descriptor-based QSAR (Random Forest)

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, rdMolDescriptors, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


## 1. Molecular Representations & Descriptors
Constructing molecules from SMILES, computing physicochemical descriptors and Morgan fingerprints.

In [2]:
# Drug-like molecules: aspirin, ibuprofen, caffeine, paracetamol, naproxen, diclofenac, celecoxib, rofecoxib
drug_data = [
    ('Aspirin', 'CC(=O)OC1=CC=CC=C1C(=O)O', 1),
    ('Ibuprofen', 'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O', 1),
    ('Caffeine', 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C', 0),
    ('Paracetamol', 'CC(=O)NC1=CC=C(O)C=C1', 1),
    ('Naproxen', 'COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O', 1),
    ('Diclofenac', 'OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl', 1),
    ('Celecoxib', 'CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F', 1),
    ('Rofecoxib', 'CS(=O)(=O)C1=CC=C(C=C1)C1=C(C(=O)OC1)C1=CC=CC=C1', 1),
    ('Metformin', 'CN(C)C(=N)NC(=N)N', 0),
    ('Theophylline', 'CN1C2=C(C(=O)N(C1=O)C)NC=N2', 0),
    ('Warfarin', 'CC(=O)CC(C1=CC=CC=C1)C1=C(O)C2=CC=CC=C2OC1=O', 1),
    ('Indomethacin', 'COC1=CC2=C(C=C1)C(C)=C(CC(=O)O)N2C(=O)C1=CC=C(Cl)C=C1', 1),
]

mols = [(name, Chem.MolFromSmiles(smi), smi, label) for name, smi, label in drug_data]
print(f'Parsed {len(mols)} molecules')

# Compute descriptors
records = []
for name, mol, smi, label in mols:
    records.append({
        'Name': name,
        'MW': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'HBA': Descriptors.NumHAcceptors(mol),
        'HBD': Descriptors.NumHDonors(mol),
        'TPSA': Descriptors.TPSA(mol),
        'RotBonds': Descriptors.NumRotatableBonds(mol),
        'Rings': Descriptors.RingCount(mol),
        'COX_inhibitor': label,
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

Parsed 12 molecules
        Name      MW     LogP  HBA  HBD  TPSA  RotBonds  Rings  COX_inhibitor
     Aspirin 180.159  1.31010    3    1 63.60         2      1              1
   Ibuprofen 206.285  3.07320    1    1 37.30         4      1              1
    Caffeine 194.194 -1.02930    3    0 61.82         0      2              0
 Paracetamol 151.165  1.35060    2    2 49.33         1      1              1
    Naproxen 218.252  2.43390    2    1 46.53         3      2              1
  Diclofenac 296.153  4.36410    2    2 49.33         4      2              1
   Celecoxib 381.379  3.51392    3    1 77.98         3      3              1
   Rofecoxib 314.362  2.55770    4    0 60.44         3      3              1
   Metformin 129.167 -1.03416    2    4 88.99         0      0              0
Theophylline 180.167 -1.03970    3    1 72.68         0      2              0
    Warfarin 308.333  3.60960    4    1 67.51         4      3              1
Indomethacin 357.793  3.92732    3    1 68.5

## 2. Morgan Fingerprints & Tanimoto Similarity Matrix

In [3]:
# Generate Morgan fingerprints (ECFP4 equivalent: radius=2, 2048 bits)
fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) for _, mol, _, _ in mols]
names = [name for name, _, _, _ in mols]

# Tanimoto similarity matrix
n = len(fps)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(names, fontsize=8)
plt.colorbar(im, label='Tanimoto Similarity')
ax.set_title('Morgan FP (ECFP4) Tanimoto Similarity Matrix')
plt.tight_layout()
plt.show()

# Most similar pair (excluding self)
np.fill_diagonal(sim_matrix, 0)
i, j = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f'Most similar pair: {names[i]} <-> {names[j]} (Tanimoto = {sim_matrix[i,j]:.3f})')

Most similar pair: Caffeine <-> Theophylline (Tanimoto = 0.457)


[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator


## 3. Murcko Scaffold Analysis

In [4]:
scaffolds = {}
for name, mol, smi, _ in mols:
    core = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_smi = Chem.MolToSmiles(core)
    if scaffold_smi not in scaffolds:
        scaffolds[scaffold_smi] = []
    scaffolds[scaffold_smi].append(name)

print(f'Unique Murcko scaffolds: {len(scaffolds)}')
for scaffold, members in sorted(scaffolds.items(), key=lambda x: -len(x[1])):
    print(f'  {scaffold:40s} -> {members}')

Unique Murcko scaffolds: 9
  c1ccccc1                                 -> ['Aspirin', 'Ibuprofen', 'Paracetamol']
  O=c1[nH]c(=O)c2[nH]cnc2[nH]1             -> ['Caffeine', 'Theophylline']
  C1=CC2=CCC=CC2=C1                        -> ['Naproxen']
  c1ccc(Nc2ccccc2)cc1                      -> ['Diclofenac']
  c1ccc(-c2ccnn2-c2ccccc2)cc1              -> ['Celecoxib']
  O=C1OCC(c2ccccc2)=C1c1ccccc1             -> ['Rofecoxib']
                                           -> ['Metformin']
  O=c1oc2ccccc2cc1Cc1ccccc1                -> ['Warfarin']
  O=C(c1ccccc1)n1ccc2ccccc21               -> ['Indomethacin']


## 4. Descriptor-Based QSAR: COX Inhibitor Classification
Random Forest on physicochemical descriptors to classify COX inhibitors vs non-inhibitors.

In [5]:
# Extended dataset with synthetic negatives for a real QSAR demo
np.random.seed(42)
extra_smiles_active = [
    'OC(=O)CC1=CC=CC=C1NC1=CC=CC=C1',  # mefenamic acid analog
    'CC1=CC=C(C=C1)C(=O)C1=CC=CC=C1O',  # ketoprofen-like
    'CC(C(=O)O)C1=CC=C(CC2=CC=CC=C2)C=C1',  # flurbiprofen-like
    'OC(=O)C1CCC2=CC(=CC=C12)F',  # fluorinated NSAID
    'CC(C(=O)O)C1=CC2=CC=CC=C2C=C1',  # naproxen analog
]
extra_smiles_inactive = [
    'C1CCCCC1',  # cyclohexane
    'CCCCCCCC',  # octane
    'C1=CC=CC=C1',  # benzene
    'CC(=O)OCC',  # ethyl acetate
    'CCCCCCCCCCCC',  # dodecane
    'OC1=CC=CC=C1',  # phenol
    'CC(C)O',  # isopropanol
    'CCOCC',  # diethyl ether
]

all_data = []
for name, mol, smi, label in mols:
    all_data.append((mol, label))
for smi in extra_smiles_active:
    m = Chem.MolFromSmiles(smi)
    if m: all_data.append((m, 1))
for smi in extra_smiles_inactive:
    m = Chem.MolFromSmiles(smi)
    if m: all_data.append((m, 0))

# Feature matrix: physicochemical + fingerprint bits
def mol_to_features(mol):
    desc = [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.NumHAcceptors(mol), Descriptors.NumHDonors(mol),
        Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol), Descriptors.FractionCSP3(mol),
        Descriptors.NumAromaticRings(mol),
    ]
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=256)
    arr = np.zeros(256)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return np.concatenate([desc, arr])

X = np.array([mol_to_features(m) for m, _ in all_data])
y = np.array([l for _, l in all_data])

print(f'Dataset: {X.shape[0]} molecules, {X.shape[1]} features')
print(f'Class balance: {sum(y)} active, {len(y)-sum(y)} inactive')

# Cross-validated RF
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_features='sqrt')
scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'\n5-fold CV Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

# Feature importance (top descriptors)
rf.fit(X, y)
desc_names = ['MW', 'LogP', 'HBA', 'HBD', 'TPSA', 'RotBonds', 'Rings', 'FracCSP3', 'AromaticRings']
desc_importances = rf.feature_importances_[:len(desc_names)]
fp_importance_total = rf.feature_importances_[len(desc_names):].sum()

fig, ax = plt.subplots(figsize=(8, 4))
imp_data = list(zip(desc_names, desc_importances)) + [('FP_bits (sum)', fp_importance_total)]
imp_data.sort(key=lambda x: x[1], reverse=True)
ax.barh([x[0] for x in imp_data], [x[1] for x in imp_data], color='#2563eb')
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importances for COX Inhibitor QSAR')
plt.tight_layout()
plt.show()

Dataset: 25 molecules, 265 features
Class balance: 14 active, 11 inactive


[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerator
[20:10:14] DEPRECATION WARNING: please use MorganGenerat


5-fold CV Accuracy: 0.920 +/- 0.098


## 5. Lipinski Rule-of-Five Filter

In [6]:
def lipinski_check(mol):
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hba = Descriptors.NumHAcceptors(mol)
    hbd = Descriptors.NumHDonors(mol)
    violations = sum([mw > 500, logp > 5, hba > 10, hbd > 5])
    return violations, mw, logp, hba, hbd

print(f'{"Name":15s} {"MW":>7s} {"LogP":>6s} {"HBA":>4s} {"HBD":>4s} {"Violations":>10s}')
print('-' * 52)
for name, mol, smi, _ in mols:
    v, mw, logp, hba, hbd = lipinski_check(mol)
    flag = ' FAIL' if v > 1 else ''
    print(f'{name:15s} {mw:7.1f} {logp:6.2f} {hba:4d} {hbd:4d} {v:10d}{flag}')

Name                 MW   LogP  HBA  HBD Violations
----------------------------------------------------
Aspirin           180.2   1.31    3    1          0
Ibuprofen         206.3   3.07    1    1          0
Caffeine          194.2  -1.03    3    0          0
Paracetamol       151.2   1.35    2    2          0
Naproxen          218.3   2.43    2    1          0
Diclofenac        296.2   4.36    2    2          0
Celecoxib         381.4   3.51    3    1          0
Rofecoxib         314.4   2.56    4    0          0
Metformin         129.2  -1.03    2    4          0
Theophylline      180.2  -1.04    3    1          0
Warfarin          308.3   3.61    4    1          0
Indomethacin      357.8   3.93    3    1          0
